# NLP04 프로젝트: mini BERT 만들기

| 항목 | 내용 |
|------|------|
| **목표** | vocab_size=8000, 파라미터 ~1M의 mini BERT 사전학습 (MLM + NSP) |
| **환경** | RunPod (Ubuntu, RTX L4 GPU) |
| **데이터** | 한글 나무위키 코퍼스 (kowiki dump → corpus.txt) |
| **모델** | d_model=128, num_heads=4, num_layers=3, d_ff=256 |
| **학습** | AdamW + WarmupLinearSchedule, 10 epochs/실험 × 4 실험 |

## 파이프라인
```
01_build_tokenizer.py  →  SentencePiece BPE 학습 (vocab_size=8000)
02_preprocess_data.py  →  MLM/NSP 전처리 → np.memmap 저장
03_pretrain.py         →  mini BERT 구현 및 사전학습
```

## 폴더 구조
```
NLP04/miniBERT/
├── data/processed/   ← spm.model, memmap 파일들
├── models/           ← 체크포인트, experiments_log.json, 그래프
├── scripts/          ← 01~03 스크립트, experiment_tracker.py
└── config.json
```

In [ ]:
import os
os.chdir('/workspace/NLP/NLP04/miniBERT')

import sys
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch

print('=== 라이브러리 버전 확인 ===')
print(f'Python     : {sys.version.split()[0]}')
print(f'PyTorch    : {torch.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'Pandas     : {pd.__version__}')
print(f'Matplotlib : {matplotlib.__version__}')

try:
    import sentencepiece as spm
    print(f'SentencePiece: {spm.__version__}')
except Exception:
    print('SentencePiece: not installed')

print()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## 1. Tokenizer 준비

### SentencePiece BPE
한국어 나무위키 코퍼스로 **Byte Pair Encoding(BPE)** 토크나이저를 학습했습니다.

| Special Token | ID | 역할 |
|---|---|---|
| `[PAD]` | 0 | 패딩 토큰 |
| `[UNK]` | 1 | 미등록 토큰 |
| `[CLS]` | 2 | 문장 시작 / NSP 분류 토큰 |
| `[SEP]` | 3 | 문장 구분자 |
| `[MASK]` | 4 | MLM 마스킹 토큰 |

- **vocab_size** = 8,000
- **character_coverage** = 0.9995 (한국어 포함 유니코드 커버리지)
- 학습 데이터: kowiki corpus.txt (약 500만 문장)

In [ ]:
import os
os.chdir('/workspace/NLP/NLP04/miniBERT')

import sentencepiece as spm

SPM_MODEL = '/workspace/NLP/NLP04/miniBERT/data/processed/spm.model'

sp = spm.SentencePieceProcessor()
sp.Load(SPM_MODEL)

# ── Special Token ID 검증
print('=== Special Token ID 검증 ===')
SPECIAL = [('[PAD]', 0), ('[UNK]', 1), ('[CLS]', 2), ('[SEP]', 3), ('[MASK]', 4)]
all_ok = True
for name, expected in SPECIAL:
    actual = sp.PieceToId(name)
    status = 'OK' if actual == expected else f'MISMATCH (got {actual})'
    print(f'  {name:8s}  expected={expected}  actual={actual}  [{status}]')
    if actual != expected:
        all_ok = False
print(f'\n  vocab_size : {sp.GetPieceSize():,}')
print(f'  검증 결과  : {"ALL PASS" if all_ok else "FAIL"}')

# ── 샘플 토크나이징
print('\n=== 샘플 문장 토크나이징 ===')
samples = [
    '안녕하세요. 자연어 처리 미니 BERT 사전학습 프로젝트입니다.',
    '한국어 위키피디아 코퍼스로 학습한 토크나이저입니다.',
    'BERT는 Bidirectional Encoder Representations from Transformers의 약자입니다.',
]
for sent in samples:
    pieces = sp.EncodeAsPieces(sent)
    ids    = sp.EncodeAsIds(sent)
    print(f'  입력  : {sent}')
    print(f'  토큰  : {pieces}')
    print(f'  ID    : {ids}')
    print(f'  복원  : {sp.Decode(ids)}')
    print()

---
## 2. 데이터 전처리 (MLM + NSP)

### Masked Language Modeling (MLM)
원본 BERT 논문과 동일한 마스킹 전략을 적용합니다.

```
전체 토큰 중 15% 선택
  ├── 80% → [MASK] 토큰으로 교체
  ├── 10% → 랜덤 토큰으로 교체
  └── 10% → 원본 토큰 유지 (변화 없음)

mlm_labels: 선택된 위치만 실제 ID, 나머지는 -100 (CrossEntropyLoss ignore)
```

### Next Sentence Prediction (NSP)
문장 쌍의 연속성을 학습합니다.

```
50%  IsNext  (1) : 실제 연속된 문장 쌍
50%  NotNext (0) : 코퍼스에서 랜덤으로 선택한 두 번째 문장

포맷: [CLS] 문장A [SEP] 문장B [SEP]
segment_ids: 문장A 위치=0, 문장B 위치=1
```

### 저장 형식 (np.memmap)
| 파일 | dtype | shape | 설명 |
|------|-------|-------|------|
| `input_ids.dat` | int32 | (N, 128) | 마스킹 적용된 입력 |
| `segment_ids.dat` | int32 | (N, 128) | 세그먼트 ID (0/1) |
| `attention_mask.dat` | int32 | (N, 128) | 패딩 마스크 (1=유효, 0=패딩) |
| `mlm_labels.dat` | int32 | (N, 128) | MLM 정답 레이블 |
| `nsp_label.dat` | int32 | (N,) | NSP 레이블 (0 또는 1) |

In [ ]:
import os
os.chdir('/workspace/NLP/NLP04/miniBERT')

import numpy as np
import json
import sentencepiece as spm

PROC_DIR  = '/workspace/NLP/NLP04/miniBERT/data/processed'
SPM_MODEL = '/workspace/NLP/NLP04/miniBERT/data/processed/spm.model'

# ── 메타데이터 로드
with open(os.path.join(PROC_DIR, 'dataset_meta.json')) as f:
    meta = json.load(f)
N   = meta['n_samples']
L   = meta['max_seq_len']

print('=== 데이터셋 정보 ===')
print(f'  샘플 수      : {N:,}')
print(f'  max_seq_len  : {L}')
print(f'  vocab_size   : {meta["vocab_size"]}')

# ── memmap 로드 (read-only)
input_ids      = np.memmap(os.path.join(PROC_DIR, 'input_ids.dat'),      dtype='int32', mode='r', shape=(N, L))
segment_ids    = np.memmap(os.path.join(PROC_DIR, 'segment_ids.dat'),    dtype='int32', mode='r', shape=(N, L))
attention_mask = np.memmap(os.path.join(PROC_DIR, 'attention_mask.dat'), dtype='int32', mode='r', shape=(N, L))
mlm_labels     = np.memmap(os.path.join(PROC_DIR, 'mlm_labels.dat'),     dtype='int32', mode='r', shape=(N, L))
nsp_label      = np.memmap(os.path.join(PROC_DIR, 'nsp_label.dat'),      dtype='int32', mode='r', shape=(N,))

print()
print('=== memmap 배열 shape ===')
for name, arr in [('input_ids', input_ids), ('segment_ids', segment_ids),
                  ('attention_mask', attention_mask), ('mlm_labels', mlm_labels),
                  ('nsp_label', nsp_label)]:
    mb = arr.nbytes / 1e6
    print(f'  {name:<18} shape={str(arr.shape):<16} dtype={arr.dtype}  {mb:.1f} MB')

# NSP 분포
n_next     = int((nsp_label[:] == 1).sum())
n_not_next = int((nsp_label[:] == 0).sum())
print(f'\n  NSP 분포  IsNext={n_next:,} ({n_next/N*100:.1f}%)  NotNext={n_not_next:,} ({n_not_next/N*100:.1f}%)')

# ── 샘플 1개 디코딩
sp = spm.SentencePieceProcessor()
sp.Load(SPM_MODEL)

PAD_ID, CLS_ID, SEP_ID, MASK_ID = 0, 2, 3, 4

print('\n=== 샘플 디코딩 (idx=0) ===')
idx       = 0
ids       = input_ids[idx].tolist()
segs      = segment_ids[idx].tolist()
mask      = attention_mask[idx].tolist()
labels    = mlm_labels[idx].tolist()
nsp       = int(nsp_label[idx])

# 유효 길이 (padding 전까지)
valid_len = sum(mask)
ids_valid = ids[:valid_len]
segs_valid = segs[:valid_len]
labels_valid = labels[:valid_len]

# SEP 기준으로 문장A/B 분리
sep_positions = [i for i, t in enumerate(ids_valid) if t == SEP_ID]
sep1 = sep_positions[0] if len(sep_positions) > 0 else valid_len

sent_a_ids = ids_valid[1:sep1]          # [CLS] 제외
sent_b_ids = ids_valid[sep1+1:-1]       # 두 번째 [SEP] 제외

print(f'  NSP 레이블    : {nsp}  ({"IsNext" if nsp == 1 else "NotNext"})')
print(f'  유효 토큰 수  : {valid_len}')
print(f'  문장A 토큰 수 : {len(sent_a_ids)}')
print(f'  문장B 토큰 수 : {len(sent_b_ids)}')
print(f'  문장A 복원    : {sp.Decode([t for t in sent_a_ids if t not in (PAD_ID,)])}')
print(f'  문장B 복원    : {sp.Decode([t for t in sent_b_ids if t not in (PAD_ID,)])}')

# 마스킹된 위치 표시
masked_pos = [i for i, (t, lb) in enumerate(zip(ids_valid, labels_valid)) if lb != -100]
print(f'\n  마스킹된 위치 수: {len(masked_pos)}')
print('  (위치, 입력토큰, 정답레이블)')
for pos in masked_pos[:8]:
    inp_tok  = sp.IdToPiece(ids_valid[pos])
    ans_tok  = sp.IdToPiece(labels_valid[pos])
    seg      = segs_valid[pos]
    print(f'    pos={pos:3d}  input=[{inp_tok}]  label=[{ans_tok}]  seg={seg}')

---
## 3. BERT 모델 구조

### 모델 아키텍처

```
BERTForPretraining
├── BERTModel
│   ├── BERTEmbedding
│   │   ├── Token Embedding     (vocab_size × d_model)
│   │   ├── Position Embedding  (max_seq_len × d_model)  ← 학습 가능
│   │   ├── Segment Embedding   (2 × d_model)
│   │   └── LayerNorm + Dropout
│   ├── TransformerEncoderLayer × num_layers
│   │   ├── MultiHeadAttention (num_heads개 병렬)
│   │   │   ├── Q/K/V Projection
│   │   │   └── Scaled Dot-Product Attention
│   │   ├── Add & LayerNorm
│   │   ├── FFN  (d_model → d_ff → d_model, GELU 활성화)
│   │   └── Add & LayerNorm
│   └── Pooler  ([CLS] token → Linear + Tanh)
├── MLM Head  (d_model → d_model → GELU → LayerNorm → vocab_size)
└── NSP Head  (d_model → 2)
```

### 하이퍼파라미터

| 파라미터 | 값 | 설명 |
|---|---|---|
| vocab_size | 8,000 | 어휘 크기 |
| d_model | 128 | Transformer 차원 |
| num_heads | 4 | Multi-Head Attention 헤드 수 |
| num_layers | 3 | Encoder 레이어 수 |
| d_ff | 256 | FFN 내부 차원 (d_model × 2) |
| max_seq_len | 128 | 최대 시퀀스 길이 |
| dropout | 0.1 | 드롭아웃 비율 |

In [ ]:
import os
os.chdir('/workspace/NLP/NLP04/miniBERT')

import sys
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── config 로드
with open('/workspace/NLP/NLP04/miniBERT/config.json') as f:
    cfg = json.load(f)

print('=== config.json ===')
for k, v in cfg.items():
    print(f'  {k:<20} : {v}')

# ── 모델 클래스 정의 (scripts/03_pretrain.py 에서 인라인)
PAD_ID = 0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_pad_mask(seq, pad_idx=PAD_ID):
    return seq == pad_idx

class GELU(nn.Module):
    def forward(self, x):
        return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))

class BERTEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len, dropout=0.1):
        super().__init__()
        self.token_emb   = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_emb     = nn.Embedding(max_seq_len, d_model)
        self.segment_emb = nn.Embedding(2, d_model)
        self.layer_norm  = nn.LayerNorm(d_model, eps=1e-12)
        self.dropout     = nn.Dropout(dropout)
        positions = torch.arange(max_seq_len).unsqueeze(0)
        self.register_buffer('positions', positions)
    def forward(self, input_ids, segment_ids):
        L   = input_ids.size(1)
        pos = self.positions[:, :L]
        x   = self.token_emb(input_ids) + self.pos_emb(pos) + self.segment_emb(segment_ids)
        return self.dropout(self.layer_norm(x))

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.d_head    = d_model // num_heads
        self.scale     = math.sqrt(self.d_head)
        self.q_proj   = nn.Linear(d_model, d_model)
        self.k_proj   = nn.Linear(d_model, d_model)
        self.v_proj   = nn.Linear(d_model, d_model)
        self.out_proj  = nn.Linear(d_model, d_model)
        self.dropout   = nn.Dropout(dropout)
    def forward(self, x, key_padding_mask=None):
        B, L, D = x.shape
        H, DH   = self.num_heads, self.d_head
        Q = self.q_proj(x).view(B, L, H, DH).transpose(1, 2)
        K = self.k_proj(x).view(B, L, H, DH).transpose(1, 2)
        V = self.v_proj(x).view(B, L, H, DH).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if key_padding_mask is not None:
            scores = scores.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf'))
        attn = self.dropout(F.softmax(scores, dim=-1))
        out  = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, L, D)
        return self.out_proj(out)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1     = nn.LayerNorm(d_model, eps=1e-12)
        self.norm2     = nn.LayerNorm(d_model, eps=1e-12)
        self.drop1     = nn.Dropout(dropout)
        self.drop2     = nn.Dropout(dropout)
        self.ffn       = nn.Sequential(nn.Linear(d_model, d_ff), GELU(),
                                       nn.Dropout(dropout), nn.Linear(d_ff, d_model))
    def forward(self, x, key_padding_mask=None):
        x = self.norm1(x + self.drop1(self.self_attn(x, key_padding_mask)))
        x = self.norm2(x + self.drop2(self.ffn(x)))
        return x

class BERTModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_len, dropout):
        super().__init__()
        self.embedding = BERTEmbedding(vocab_size, d_model, max_seq_len, dropout)
        self.encoders  = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)])
        self.pool_fc  = nn.Linear(d_model, d_model)
        self.pool_act = nn.Tanh()
    def forward(self, input_ids, segment_ids, attention_mask):
        kpm = get_pad_mask(input_ids)
        x   = self.embedding(input_ids, segment_ids)
        for enc in self.encoders:
            x = enc(x, key_padding_mask=kpm)
        pooled = self.pool_act(self.pool_fc(x[:, 0, :]))
        return x, pooled

class BERTForPretraining(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_len, dropout):
        super().__init__()
        self.bert     = BERTModel(vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_len, dropout)
        self.mlm_head = nn.Sequential(nn.Linear(d_model, d_model), GELU(),
                                      nn.LayerNorm(d_model, eps=1e-12), nn.Linear(d_model, vocab_size))
        self.nsp_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, 2))
    def forward(self, input_ids, segment_ids, attention_mask):
        seq_out, pooled = self.bert(input_ids, segment_ids, attention_mask)
        return self.mlm_head(seq_out), self.nsp_head(pooled)

# ── 모델 생성
model = BERTForPretraining(
    vocab_size  = cfg['vocab_size'],
    d_model     = cfg['d_model'],
    num_heads   = cfg['num_heads'],
    num_layers  = cfg['num_layers'],
    d_ff        = cfg['d_ff'],
    max_seq_len = cfg['max_seq_len'],
    dropout     = cfg['dropout'],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n=== 모델 파라미터 수 ===')
print(f'  Total parameters    : {total_params:>10,}')
print(f'  Trainable parameters: {train_params:>10,}')
print(f'  모델 크기 (fp32)    : {total_params * 4 / 1e6:.2f} MB')

print(f'\n=== 모델 구조 ===')
print(model)

---
## 4. Pretrain 학습 결과

### 학습 설정
- **Optimizer**: AdamW (weight_decay=0.01)
- **Scheduler**: WarmupLinearSchedule (warmup = 전체 step의 10%)
- **Loss**: MLM CrossEntropyLoss (ignore_index=-100) + NSP CrossEntropyLoss
- **Gradient Clipping**: max_norm=1.0
- **total_loss** = mlm_loss + nsp_loss

### 실험 이력
총 4번의 실험을 진행하며 하이퍼파라미터를 조정했습니다.

In [ ]:
import os
os.chdir('/workspace/NLP/NLP04/miniBERT')

import json
import pandas as pd

LOG_JSON = '/workspace/NLP/NLP04/miniBERT/models/experiments_log.json'

with open(LOG_JSON, 'r', encoding='utf-8') as f:
    experiments = json.load(f)

print(f'총 실험 횟수: {len(experiments)}')

# ── 요약 DataFrame
rows = []
for exp in experiments:
    cfg_snap = exp.get('config', {})
    rows.append({
        'exp_id'          : exp['experiment_id'],
        'timestamp'       : exp['timestamp'][:16],
        'duration_min'    : round(exp['duration_sec'] / 60, 1),
        'total_params'    : f"{exp['total_params']:,}",
        'd_model'         : cfg_snap.get('d_model'),
        'num_layers'      : cfg_snap.get('num_layers'),
        'epochs'          : cfg_snap.get('epochs'),
        'batch_size'      : cfg_snap.get('batch_size'),
        'lr'              : cfg_snap.get('learning_rate'),
        'best_epoch'      : exp['best_epoch'],
        'best_total_loss' : round(exp['best_total_loss'], 4),
        'best_mlm_loss'   : round(exp['best_mlm_loss'],   4),
        'best_nsp_loss'   : round(exp['best_nsp_loss'],   4),
        'best_mlm_acc'    : round(exp['best_mlm_acc'],    4),
        'best_nsp_acc'    : round(exp['best_nsp_acc'],    4),
        'final_mlm_acc'   : round(exp['final_mlm_acc'],   4),
        'final_nsp_acc'   : round(exp['final_nsp_acc'],   4),
    })

df = pd.DataFrame(rows).set_index('exp_id')

print('\n=== 실험 요약 ===')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
display(df)

# ── 최고 성능 실험
best_idx = df['best_total_loss'].idxmin()
print(f'\n최저 total_loss: 실험 #{best_idx}')
print(df.loc[best_idx])

In [ ]:
import os
os.chdir('/workspace/NLP/NLP04/miniBERT')

import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

MODEL_DIR = '/workspace/NLP/NLP04/miniBERT/models'

def show_png(path, title=''):
    if not os.path.exists(path):
        print(f'[skip] 파일 없음: {path}')
        return
    img = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(img)
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=12, pad=8)
    plt.tight_layout()
    plt.show()

# ── 1. 실험별 epoch 곡선 (exp01~04_curves.png)
print('=== 실험별 Loss / Accuracy 곡선 ===')
for i in range(1, 5):
    path = os.path.join(MODEL_DIR, f'exp{i:02d}_curves.png')
    show_png(path, title=f'Experiment #{i} — Loss & Accuracy')

# ── 2. 전체 실험 비교
print('\n=== 전체 실험 Best Total Loss 비교 ===')
show_png(os.path.join(MODEL_DIR, 'experiments_comparison.png'), '실험별 Best Total Loss')

# ── 3. 하이퍼파라미터 히트맵
print('\n=== 하이퍼파라미터 히트맵 ===')
show_png(os.path.join(MODEL_DIR, 'heatmap.png'), '하이퍼파라미터 vs Best Total Loss')

# ── 4. 최종 학습 Loss 그래프
print('\n=== 최종 실험 Pretrain Loss ===')
show_png(os.path.join(MODEL_DIR, 'pretrain_loss.png'), 'mini BERT Pretraining Loss (최종 실험)')

# ── 5. 학습 히스토리에서 직접 재플롯 (train_history.json)
hist_path = os.path.join(MODEL_DIR, 'train_history.json')
if os.path.exists(hist_path):
    with open(hist_path) as f:
        history = json.load(f)

    epochs     = [h['epoch']      for h in history]
    tot_losses = [h['total_loss'] for h in history]
    mlm_losses = [h['mlm_loss']   for h in history]
    nsp_losses = [h['nsp_loss']   for h in history]
    mlm_accs   = [h['mlm_acc']    for h in history]
    nsp_accs   = [h['nsp_acc']    for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    ax.plot(epochs, tot_losses, 'o-', color='steelblue',  label='Total Loss')
    ax.plot(epochs, mlm_losses, 's-', color='darkorange', label='MLM Loss')
    ax.plot(epochs, nsp_losses, '^-', color='green',      label='NSP Loss')
    ax.set_title('Loss per Epoch')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(epochs, mlm_accs, 'o-', color='darkorange', label='MLM Accuracy')
    ax.plot(epochs, nsp_accs, 's-', color='green',      label='NSP Accuracy')
    ax.set_title('Accuracy per Epoch')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.suptitle('mini BERT Pretraining — 최종 실험 학습 곡선', fontsize=13)
    fig.tight_layout()
    plt.show()

    print(f'\n  첫 epoch  — total={tot_losses[0]:.4f}  mlm={mlm_losses[0]:.4f}  nsp={nsp_losses[0]:.4f}')
    print(f'  마지막 epoch — total={tot_losses[-1]:.4f}  mlm={mlm_losses[-1]:.4f}  nsp={nsp_losses[-1]:.4f}')
    print(f'  best mlm_acc={max(mlm_accs):.4f}  best nsp_acc={max(nsp_accs):.4f}')

---
## 5. 결론

### 학습 성과

총 **4번의 실험** (총 ~30 epoch)을 통해 mini BERT 사전학습을 완료했습니다.

| 지표 | 초기 (epoch 1) | 최종 |
|------|---------------|------|
| MLM loss | ~ 초기값 | 안정적 감소 |
| NSP loss | ~ 초기값 | 안정적 감소 |
| MLM accuracy | ~ 초기값 | 향상 |
| NSP accuracy | ~ 50% (랜덤) | 향상 |

### 실험별 주요 변경사항
- **Exp 1**: 기본 설정 (d_model=128, num_layers=3, epochs=10)
- **Exp 2**: learning_rate 조정
- **Exp 3**: batch_size 조정
- **Exp 4**: 최적 설정으로 추가 학습

### 배운 점
1. **MLM 마스킹 전략**: 80/10/10 비율이 모델의 일반화에 중요
2. **NSP**: 초기에 50% (랜덤 수준)에서 시작하여 학습이 진행될수록 향상
3. **WarmupLinearSchedule**: 학습 초기 불안정성을 줄이는 데 효과적
4. **np.memmap**: 대용량 데이터를 메모리 부담 없이 처리 가능
5. **ExperimentTracker**: 실험 간 하이퍼파라미터 비교를 자동화하여 체계적 실험 관리

### 향후 과제
- 사전학습된 BERT를 fine-tuning하여 downstream 태스크 (감성 분석, NER 등) 적용
- vocab_size, d_model 확장 실험
- 더 큰 코퍼스로 재학습